# Pre-trained SchNet models on QM9

Evaluate the official pre-trained SchNet models ([Schütt et al., 2017](https://arxiv.org/abs/1706.08566))
on QM9: one model per target property. SchNet learns continuous filters over the distances between
atoms. `SchNet.from_qm9_pretrained` downloads the SchNetPack weights, converts them to Keras (reading the
files needs PyTorch) and returns the data split the model was trained with.

Same model as PyG's [`examples/qm9_pretrained_schnet.py`](https://github.com/pyg-team/pytorch_geometric/blob/master/examples/qm9_pretrained_schnet.py).

In [ ]:
# Install K3-Node, then choose a backend: "tensorflow", "torch" or "jax"
!pip install -q "k3-node[examples] @ git+https://github.com/anas-rz/k3-node"
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Load the data

In [ ]:
import keras
import numpy as np
from keras import ops
from k3_node.datasets import QM9
from k3_node.loader import DataLoader
from k3_node.models import SchNet

dataset = QM9("data/QM9")

## Evaluate every target

The mean absolute error on each model's test molecules; energies in meV.

In [ ]:
class Predictor(keras.Model):  # evaluates the pre-trained model on batches of molecules
    def __init__(self, net):
        super().__init__()
        self.net = net

    def call(self, data):
        return self.net(data.z, data.pos, data.batch, batch_size=data.num_graphs)


for target in range(12):
    model, (train_dataset, val_dataset, test_dataset) = SchNet.from_qm9_pretrained("data/QM9/pretrained", dataset, target)
    predictions = Predictor(model).predict(DataLoader(test_dataset, batch_size=64), verbose=0)[:, 0]
    y = np.concatenate([ops.convert_to_numpy(graph.y)[:, target] for graph in test_dataset])
    errors = np.abs(predictions - y)
    if target in [2, 3, 4, 6, 7, 8, 9, 10]:  # energies: report meV instead of eV
        errors = 1000 * errors
    print(f"Target {target:02d}: MAE {errors.mean():.5f} ± {errors.std():.5f}")